In [22]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.impute import SimpleImputer
from src.f1_strategy_dataset.preprocessing import preprocess_f1_data
from sklearn.metrics import roc_auc_score, precision_score
import warnings
warnings.filterwarnings('ignore')

In [23]:
# Load the F1 Strategy Dataset
df = pd.read_csv('data/f1_strategy_dataset_v4.csv')

# Display the first row
df.head(1)

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.0,0.0,0,0,0.017241,0.117647,0.0


In [24]:
# Basic statistics
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101371 entries, 0 to 101370
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  101371 non-null  object 
 1   LapNumber               101371 non-null  int64  
 2   Compound                101305 non-null  object 
 3   Stint                   101371 non-null  int64  
 4   TyreLife                101371 non-null  float64
 5   Position                101371 non-null  int64  
 6   LapTime (s)             101371 non-null  float64
 7   Race                    101371 non-null  object 
 8   Year                    101371 non-null  int64  
 9   LapTime_Delta           101371 non-null  float64
 10  Cumulative_Degradation  101371 non-null  float64
 11  PitStop                 101371 non-null  int64  
 12  PitNextLap              101371 non-null  int64  
 13  RaceProgress            101371 non-null  float64
 14  Normalized_TyreLife 

In [ ]:
# Statistical summary of the dataset
df.describe()

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
count,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000
mean,30.444841,2.046394,14.549339,9.759132,92.587188,2023.589685,-0.203891,-29.550051,0.251581,0.254797,0.432618,0.386521,-0.004636
std,18.146942,0.948797,10.313385,5.406456,33.231414,1.098518,45.344910,70.235759,0.433924,0.435749,0.258129,0.259906,3.912725
min,1.000000,1.000000,1.000000,1.000000,67.012000,2022.000000,-2403.895000,-274.564000,0.000000,0.000000,0.012821,0.012821,-18.000000
25%,15.000000,1.000000,7.000000,5.000000,82.021000,2023.000000,-7.253000,-51.054500,0.000000,0.000000,0.210526,0.172414,-2.000000
50%,30.000000,2.000000,13.000000,10.000000,91.167000,2024.000000,-0.027000,-21.678000,0.000000,0.000000,0.421053,0.333333,0.000000
75%,45.000000,3.000000,20.000000,14.000000,99.356000,2025.000000,5.705000,-3.725500,1.000000,1.000000,0.631579,0.562500,2.000000
max,78.000000,8.000000,78.000000,20.000000,2526.253000,2025.000000,2433.472000,2412.431000,1.000000,1.000000,1.000000,1.000000,18.000000


In [7]:
# Missing values
df.isnull().sum()

Driver                     0
LapNumber                  0
Compound                  66
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
Race                       0
Year                       0
LapTime_Delta              0
Cumulative_Degradation     0
PitStop                    0
PitNextLap                 0
RaceProgress               0
Normalized_TyreLife        0
Position_Change            0
dtype: int64

In [8]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
df['Compound'] = df['Compound'].astype(dtype)

# Verify the conversion
df['Compound'].cat.categories.tolist()

['SOFT', 'MEDIUM', 'HARD']

In [9]:
df['Compound'].cat.codes.value_counts()

 2    44967
 1    37639
 0    12744
-1     6021
Name: count, dtype: int64

In [10]:
# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)

PitNextLap
0    0.745203
1    0.254797
Name: proportion, dtype: float64

In [ ]:
# Create lagged features for Position, LapTime, and degradation metrics
# Group by Driver, Race, Year to maintain temporal order within sequences

# Create lagged features
df['Position_lag1'] = df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(1)
df['Position_lag2'] = df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(2)
df['Position_lag3'] = df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(3)

# LapTime lags
df['LapTime_lag1'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(1)
df['LapTime_lag2'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(2)
df['LapTime_lag3'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(3)

# Rolling average pace (using LapTime_Delta as a proxy for pace change)
# Rolling average of LapTime_Delta over 3 laps
df['Rolling_avg_pace'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Rolling degradation (rolling average of Cumulative_Degradation)
df['Rolling_degradation'] = df.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Fill NaN values (from shifts) with 0
# Only fill numeric columns with 0, leave string/categorical columns unchanged
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)


In [ ]:
# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
df['Compound'] = imputer.fit_transform(df['Compound'].values.reshape(-1, 1)).flatten()

In [ ]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
df['Compound_encoded'] = encoder.fit_transform(df['Compound'].to_numpy().reshape(-1, 1)).flatten()

# Drop the original Compound column
df.drop('Compound', axis=1, inplace=True)